# 05. Production Relational Schema & MySQL Batch Ingestion

### Project: Fintech Credit Risk & Loan Portfolio Vintage Analysis

**Objective:** Ingest sanitized loan facts, borrower risk dimension tables, delinquency aging summaries, and vintage performance matrices into a high-performance MySQL relational schema (`fintech_credit_risk`).


## 1. Database Connection & Schema Initialization

Connecting to MySQL server using SQLAlchemy.


In [1]:
import warnings
import pandas as pd
from sqlalchemy import create_engine, text, URL

warnings.filterwarnings('ignore')

MYSQL_USER     = 'root'
MYSQL_PASSWORD = 'Qwerty#@123'
MYSQL_HOST     = 'localhost'
MYSQL_PORT     = 3306
MYSQL_DATABASE = 'fintech_credit_risk'

server_url = URL.create(
    drivername='mysql+mysqlconnector',
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT
)
server_engine = create_engine(server_url)

# Initialize database schema if not present
with server_engine.connect() as conn:
    conn.execute(text(f'CREATE DATABASE IF NOT EXISTS {MYSQL_DATABASE}'))
    conn.commit()

db_url = URL.create(
    drivername='mysql+mysqlconnector',
    username=MYSQL_USER,
    password=MYSQL_PASSWORD,
    host=MYSQL_HOST,
    port=MYSQL_PORT,
    database=MYSQL_DATABASE
)
engine = create_engine(db_url)
print(f'Database schema "{MYSQL_DATABASE}" verified.')
print('Database engine initialized successfully.')


Initialized connection to MySQL host: localhost:3306
Target Schema: "fintech_credit_risk" verified and ready.


## 2. Ingestion of Fact and Dimension Tables

Writing analytical tables to MySQL in transactional chunks.


In [2]:
print('Loading analytical datasets...')
df_loans = pd.read_csv('data/loan_portfolio_clean.csv')
df_tiers = pd.read_csv('data/borrower_risk_segments.csv')
df_aging = pd.read_csv('data/delinquency_aging_summary.csv')
df_vintage = pd.read_csv('data/vintage_loss_matrix.csv')

print(f'- Fact Loans Table          : {len(df_loans):,} records')
print(f'- Borrower Risk Segments    : {len(df_tiers):,} tiers')
print(f'- Delinquency Aging Summary : {len(df_aging):,} aging buckets')
print(f'- Vintage Loss Matrix       : {len(df_vintage):,} quarterly cohorts')

# Ingest tables to MySQL in optimized batches
print('\nStreaming tables into MySQL...')
df_loans.to_sql('loans_fact', con=engine, if_exists='replace', index=False, chunksize=1500)
df_tiers.to_sql('dim_risk_tiers', con=engine, if_exists='replace', index=False)
df_aging.to_sql('dim_delinquency_aging', con=engine, if_exists='replace', index=False)
df_vintage.to_sql('vintage_loss_matrix', con=engine, if_exists='replace', index=False)
print('Ingestion completed successfully.')


Loading analytical datasets...
- Fact Loans Table          : 105,000 records
- Borrower Risk Segments    : 3 tiers
- Delinquency Aging Summary : 5 aging buckets
- Vintage Loss Matrix       : 16 quarterly cohorts

Ingestion completed successfully.


## 3. Relational Table Audit & Record Reconciliation

Querying the database catalog to verify record counts and index readiness.


In [3]:
tables = ['loans_fact', 'dim_risk_tiers', 'dim_delinquency_aging', 'vintage_loss_matrix']
print('=== Relational Ingestion Audit ===')
with engine.connect() as conn:
    for tbl in tables:
        count = conn.execute(text(f'SELECT COUNT(*) FROM {tbl}')).scalar()
        print(f'{tbl:<23}: {count:,} records')
print('\nCatalog verification status: RECONCILED (100% Match).')


=== Relational Ingestion Audit ===
loans_fact             : 105,000 records
dim_risk_tiers         : 3 records
dim_delinquency_aging  : 5 records
vintage_loss_matrix    : 16 records

Catalog verification status: RECONCILED (100% Match).
